# **Libraries**

In [ ]:
%run nb_sp_common

# **Connections**

Inventory Fabric data connections and manage access to them via the
[Connections REST API](https://learn.microsoft.com/rest/api/fabric/core/connections).

**Permission boundary (read this first).** The Connections API has *no*
self-escalation path. The calling identity - here the service principal
loaded by `nb_sp_common` - can only:

| Action | Endpoint | Caller must already hold |
| --- | --- | --- |
| List connections | `GET /v1/connections` | *any* role on the connection (returns only those it can see) |
| List role assignments | `GET .../roleAssignments` | Owner on the connection, or gateway Admin |
| Grant a principal (reshare) | `POST .../roleAssignments` | **UserWithReshare / Owner**, or **gateway Admin** |
| Rotate stored credential | `PATCH /v1/connections/{id}` | permission on the connection, or gateway Admin |

You **cannot** add yourself to a connection where the SP has no role, or
only plain `User` role. There is no documented admin override. Grants of
`403 InsufficientPermissionsToManageConnection` are the API refusing an
escalation, not a bug.

**Identity model.** The SP is the *manager*. "Add my credentials" means
granting a **target principal** (default: your own Entra user object ID)
a role on connections the SP is entitled to manage.

# **Functions**

## **Connection functions**

In [ ]:
# --------------------------------------------------------
#  Fabric error surfacer
# --------------------------------------------------------
def raise_for_fabric(response: requests.Response) -> requests.Response:
    """
    Raise with the Fabric error body attached, else return the response.

    Fabric returns a JSON body with `errorCode` / `message` on failures
    (e.g. InsufficientPermissionsToManageConnection). requests'
    raise_for_status swallows that body, so surface it explicitly.

    :param response: The HTTP response to check.
    :returns: The same response when status is < 400.
    :raises requests.HTTPError: With the Fabric errorCode/message appended.
    """
    if response.status_code < 400:
        return response

    try:
        body = response.json()
        detail = f"{body.get('errorCode')}: {body.get('message')}"
    except ValueError:
        detail = response.text

    raise requests.HTTPError(
        f"{response.status_code} {response.reason} - {detail}",
        response=response,
    )


# --------------------------------------------------------
#  List connections (paginated)
# --------------------------------------------------------
def list_connections(access_token: str) -> List[Dict]:
    """
    List every connection the calling identity has a role on.

    Walks continuationToken pages to completion. The SP only sees
    connections it holds at least one role on - this is not a
    tenant-wide inventory.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :returns: List of connection objects (id, displayName, connectivityType,
        connectionDetails, credentialDetails, ...).
    :raises requests.HTTPError: If a page request fails.
    """
    endpoint = "https://api.fabric.microsoft.com/v1/connections"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    connections: List[Dict] = []
    params: Dict = {}

    while True:
        response = raise_for_fabric(
            requests.get(endpoint, headers=headers, params=params)
        )
        payload = response.json()
        connections.extend(payload.get("value", []))

        token = payload.get("continuationToken")
        if not token:
            break
        params = {"continuationToken": token}

    return connections


# --------------------------------------------------------
#  Find a connection id by display name
# --------------------------------------------------------
def get_connection_id(access_token: str, display_name: str) -> str:
    """
    Resolve a connection's ID by its display name.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param display_name: Display name of the target connection.
    :returns: Connection GUID.
    :raises ValueError: If no visible connection matches the name.
    """
    for conn in list_connections(access_token):
        if conn.get("displayName") == display_name:
            return conn["id"]

    raise ValueError(
        f"No connection named '{display_name}' is visible to this identity."
    )

## **Role assignment functions**

In [ ]:
# --------------------------------------------------------
#  List connection role assignments
# --------------------------------------------------------
def list_connection_role_assignments(
    access_token: str, connection_id: str
) -> List[Dict]:
    """
    List role assignments on a connection.

    Requires Owner on the connection or Admin on its gateway. Callers with
    a lesser role get 403 - catch it at the call site to keep an inventory
    loop going.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param connection_id: GUID of the target connection.
    :returns: List of role-assignment objects (id, principal, role).
    :raises requests.HTTPError: If the request fails (e.g. 403 without Owner).
    """
    endpoint = (
        f"https://api.fabric.microsoft.com/v1/connections/"
        f"{connection_id}/roleAssignments"
    )
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    assignments: List[Dict] = []
    params: Dict = {}

    while True:
        response = raise_for_fabric(
            requests.get(endpoint, headers=headers, params=params)
        )
        payload = response.json()
        assignments.extend(payload.get("value", []))

        token = payload.get("continuationToken")
        if not token:
            break
        params = {"continuationToken": token}

    return assignments


# --------------------------------------------------------
#  Add a connection role assignment (grant / reshare)
# --------------------------------------------------------
def add_connection_role_assignment(
    access_token: str,
    connection_id: str,
    principal_id: str,
    principal_type: str = "User",
    role: str = "User",
) -> Dict:
    """
    Grant a principal a role on a connection.

    The SP must already hold UserWithReshare or Owner on the connection, or
    be Admin on the bound gateway. There is no self-escalation: the API
    returns 403 InsufficientPermissionsToManageConnection otherwise.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param connection_id: GUID of the target connection.
    :param principal_id: Entra object ID of the principal to grant. For a
        user this is the user's object ID (not the UPN).
    :param principal_type: One of User, ServicePrincipal, Group,
        ServicePrincipalProfile, EntireTenant. Defaults to User.
    :param role: One of User, UserWithReshare, Owner. Defaults to User.
    :returns: The created role-assignment object.
    :raises requests.HTTPError: If the caller lacks reshare/owner/gateway-admin.
    """
    endpoint = (
        f"https://api.fabric.microsoft.com/v1/connections/"
        f"{connection_id}/roleAssignments"
    )
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }
    body = {
        "principal": {"id": principal_id, "type": principal_type},
        "role": role,
    }

    response = raise_for_fabric(
        requests.post(endpoint, headers=headers, json=body)
    )
    result = response.json()
    print(
        f"Granted {principal_type} {principal_id} role '{role}' "
        f"on connection {connection_id}."
    )
    return result

## **Credential rotation function (sensitive)**

In [ ]:
# --------------------------------------------------------
#  Update a cloud connection's stored credential
# --------------------------------------------------------
def update_connection_credentials(
    access_token: str,
    connection_id: str,
    credentials: Dict,
    connectivity_type: str = "ShareableCloud",
    single_sign_on_type: str = "None",
    skip_test_connection: bool = False,
) -> Dict:
    """
    Set / rotate the stored credential a cloud connection authenticates with.

    Requires permission on the connection or gateway-admin. This overwrites
    how the connection reaches its data source - treat as a rotation, not an
    additive op. Prefer a Key Vault reference over an inline secret.

    Gateway (on-premises / VNet) connections need RSA-encrypted credential
    payloads and are out of scope here - this covers cloud connectivity
    types (ShareableCloud, PersonalCloud) only.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param connection_id: GUID of the target connection.
    :param credentials: A Credentials object, e.g.
        {"credentialType": "Basic", "username": "...", "password": "..."} or
        {"credentialType": "ServicePrincipal", "tenantId": "...",
         "servicePrincipalClientId": "...",
         "servicePrincipalSecretReference": {"connectionId": "<kv-conn>",
                                             "secretName": "..."}}.
    :param connectivity_type: ShareableCloud or PersonalCloud.
    :param single_sign_on_type: SSO type; None for stored-credential auth.
    :param skip_test_connection: When True, skip the live test on update.
    :returns: The updated connection object.
    :raises requests.HTTPError: On insufficient permission or bad credentials.
    """
    endpoint = f"https://api.fabric.microsoft.com/v1/connections/{connection_id}"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }
    body = {
        "connectivityType": connectivity_type,
        "credentialDetails": {
            "singleSignOnType": single_sign_on_type,
            "skipTestConnection": skip_test_connection,
            "credentials": credentials,
        },
    }

    response = raise_for_fabric(
        requests.patch(endpoint, headers=headers, json=body)
    )
    print(f"Updated credentials on connection {connection_id}.")
    return response.json()

# **Operation**

## **Inventory connections**

Lists every connection the SP can see, and probes each for role
assignments. A `403` on the probe means the SP lacks Owner/gateway-admin
on that connection - it is reported as `manageable=False`, which is
exactly the set you *cannot* grant yourself onto.

In [ ]:
# --------------------------------------------------------
#  Enumerate visible connections + manageability
# --------------------------------------------------------
connections = list_connections(access_token)
print(f"Visible connections: {len(connections)}\n")

for conn in connections:
    conn_id = conn["id"]
    name = conn.get("displayName", "(no name)")
    ctype = conn.get("connectivityType")
    details = conn.get("connectionDetails", {})
    cred_type = conn.get("credentialDetails", {}).get("credentialType")

    # A successful roleAssignments read implies Owner / gateway-admin,
    # i.e. the SP can grant on this connection.
    try:
        assignments = list_connection_role_assignments(access_token, conn_id)
        manageable = True
    except requests.HTTPError:
        assignments = []
        manageable = False

    print(f"- {name}  [{ctype}]")
    print(f"    id:           {conn_id}")
    print(f"    source:       {details.get('type')} / {details.get('path')}")
    print(f"    credential:   {cred_type}")
    print(f"    manageable:   {manageable}  ({len(assignments)} role assignment(s))")

## **Grant a principal access (reshare)**

Adds `<TargetPrincipalObjectId>` to a connection the SP can manage.
Default target is a **user** object ID (your own, to "add my
credentials"); switch `principal_type` to `ServicePrincipal` or `Group`
as needed. Find a user's object ID in Entra ID > Users, or via
`az ad user show --id <upn> --query id -o tsv`.

In [ ]:
# --------------------------------------------------------
#  Grant parameters
# --------------------------------------------------------
target_connection_name = "<ConnectionDisplayName>"
target_principal_id = "<TargetPrincipalObjectId>"  # Entra object ID
target_principal_type = "User"                     # User | ServicePrincipal | Group
target_role = "User"                               # User | UserWithReshare | Owner

# --------------------------------------------------------
#  Grant
# --------------------------------------------------------
connection_id = get_connection_id(access_token, target_connection_name)

add_connection_role_assignment(
    access_token,
    connection_id,
    principal_id=target_principal_id,
    principal_type=target_principal_type,
    role=target_role,
)

# --------------------------------------------------------
#  Verify
# --------------------------------------------------------
for ra in list_connection_role_assignments(access_token, connection_id):
    principal = ra.get("principal", {})
    print(f"{ra.get('role'):16} {principal.get('type'):18} {principal.get('id')}")

## **Rotate a stored credential (optional, sensitive)**

Overwrites the credential a cloud connection authenticates with. Left
guarded behind `ROTATE = False` so a run-all doesn't fire it. Prefer a
Key Vault secret reference over an inline secret; never commit a real
secret value.

In [ ]:
# --------------------------------------------------------
#  Rotation guard - flip to True only when you mean it
# --------------------------------------------------------
ROTATE = False

target_connection_name = "<ConnectionDisplayName>"

# Example: Basic (username / password). Swap for a ServicePrincipal or a
# Key Vault-referenced secret in production.
new_credentials = {
    "credentialType": "Basic",
    "username": "<Username>",
    "password": "<Password>",
}

if ROTATE:
    connection_id = get_connection_id(access_token, target_connection_name)
    update_connection_credentials(
        access_token,
        connection_id,
        credentials=new_credentials,
        connectivity_type="ShareableCloud",
    )
else:
    print("ROTATE is False - no credential was changed.")